In [ ]:
import torch
import os
from safetensors.torch import load_file
from diffusion.model.autoencoder import Autoencoder
from diffusion.model.clip import CLIP
from diffusion.model.unet import UNet

In [2]:
# load weights
weights_path = os.path.join("..", "data", "weights", "v1-5-pruned-emaonly.safetensors")
state_dict = load_file(weights_path, device='cpu')

In [3]:
# split state dict into models
vae_sd = {}
clip_sd = {}
unet_sd = {}

for k, v in state_dict.items():
    if k.startswith('first_stage_model.'):
        new_k = k.replace('first_stage_model.', '')
        vae_sd[new_k] = v
    elif k.startswith('cond_stage_model.transformer.'):
        new_k = k.replace('cond_stage_model.transformer.', '')
        clip_sd[new_k] = v
    elif k.startswith('model.diffusion_model.'):
        new_k = k.replace('model.diffusion_model.', '')
        unet_sd[new_k] = v


In [4]:
# intantiate my model
my_vae = Autoencoder()
my_clip = CLIP()
my_unet = UNet()

# VAE

In [5]:
print(f'Weights and biases in my VAE: {len(my_vae.state_dict())}')
print(f'Weights and biases in SD1.5 ckpt: {len(vae_sd)}')

Weights and biases in my VAE: 248
Weights and biases in SD1.5 ckpt: 248


# CLIP

In [6]:
print(f'Weights and biases in my CLIP text encoder: {len(my_clip.state_dict())}')
print(f'Weights and biases in SD1.5 ckpt: {len(clip_sd)}')

Weights and biases in my CLIP text encoder: 196
Weights and biases in SD1.5 ckpt: 197


SD1.5's CLIP state dict includes one extra entry, a registered buffer for token position indices, that I didn't use

# UNet

In [7]:
print(f'Weights and biases in my UNet: {len(my_unet.state_dict())}')
print(f'Weights and biases in SD1.5 ckpt: {len(unet_sd)}')

Weights and biases in my UNet: 686
Weights and biases in SD1.5 ckpt: 686


---

# LOAD ALL MODELS

In [ ]:
from diffusion.convert_weights import load_all_models

load_all_models(weights_path, unet=my_unet, vae=my_vae, clip=my_clip)